# Modelagem e Avaliação de Modelos de Ensemble

### Objetivo

- Treinar modelos de ensemble (Random Forest e Gradient Boosting) reutilizando a **mesma** metodologia do notebook de MLP (mesmo pré-processador, mesma validação cruzada e mesmas métricas), para que a comparação seja justa
- Aplicar validação cruzada estratificada
- Tratar o desbalanceamento de classes (threshold ajustado, `class_weight` e SMOTE)
- Analisar o trade-off de custo entre falso positivo e falso negativo
- Registrar todos os experimentos no **MLflow**
- Consolidar os resultados numa tabela comparativa junto com o MLP e escolher o modelo campeão
- Exportar o modelo final e os resultados

### Metas

Mesmas metas técnicas definidas na Etapa 2:

- ROC-AUC maior ou igual a 0,80
- F1-score maior ou igual a 0,60
- recall maior ou igual a 0,55

## Sumário
- [Importando Bibliotecas](#Importando-Bibliotecas)
- [Pré-processamento](#preprocessamento)
    - [Separando Features e Target](#Separando-Features-e-Target)
    - [Separando dados Numéricos e Categóricos](#num-cat)
- [Construindo os Pipelines](#Construindo-os-Pipelines)
    - [Validação Cruzada](#cv)
    - [Funções de Desempenho](#desempenho)
- [MLflow](#mlflow)
- [Random Forest](#rf)
    - [Baseline](#rf-baseline)
    - [Threshold ajustado](#rf-threshold)
    - [Class weight](#rf-classweight)
    - [SMOTE](#rf-smote)
- [Gradient Boosting](#gb)
    - [Baseline](#gb-baseline)
    - [Threshold ajustado](#gb-threshold)
    - [SMOTE](#gb-smote)
- [Comparação Consolidada](#comparacao)
    - [Análise de Custo (FP × FN)](#custo)
- [Conclusão](#conclusao)
    - [Recomendação](#recomendacao)
    - [Exportação](#exportacao)
        - [Modelo](#Modelo)
        - [Resultados](#Resultados)

<h2 id="Importando-Bibliotecas">Importando bibliotecas</h2>

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import joblib

import mlflow

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

Salvando `RANDOM_STATE` para permitir reprodutibilidade e localizando a raiz do projeto (mesma lógica do notebook do MLP):

In [ ]:
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().resolve()
if "notebooks" in str(PROJECT_ROOT):
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

pd.set_option("display.max_columns", None)

<h2 id="preprocessamento">Pré-processamento</h2>

Repetimos aqui as mesmas transformações do notebook do MLP para garantir que os
modelos sejam treinados sobre exatamente a mesma base. A coluna `TotalCharges`
vem como texto no CSV original e precisa ser convertida para número; os valores
que não convertem viram `NaN` e são tratados pelo imputador do pipeline.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.info()

### Separando Features e Target

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = (df["Churn"] == "Yes").astype(int)

<h3 id="num-cat">Separando dados Numéricos e Categóricos</h3>

In [ ]:
cols_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]
cols_categoricas = [col for col in X.columns if col not in cols_numericas]

<h2 id="Construindo-os-Pipelines">Construindo os Pipelines</h2>

O `preprocessador` é idêntico ao do notebook do MLP. Modelos baseados em árvore
não exigem escalonamento, mas mantemos o `StandardScaler` para preservar a
mesma matriz de features e tornar a comparação justa entre os notebooks.

In [ ]:
pipeline_numerico = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

pipeline_categorico = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessador = ColumnTransformer([
    ("num", pipeline_numerico, cols_numericas),
    ("cat", pipeline_categorico, cols_categoricas),
])

<h3 id="cv">Validação Cruzada</h3>

Mesma divisão estratificada em 5 folds usada no notebook do MLP.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

<h3 id="desempenho">Funções de Desempenho</h3>

Reaproveitamos as funções `calcular_metricas` e `cv_resultados` do notebook do
MLP. A `cv_resultados` foi estendida para também acumular a **matriz de confusão
média** entre os folds — precisaremos dela na análise de custo (FP × FN).

In [ ]:
def calcular_metricas(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

In [ ]:
def cv_resultados(pipeline, threshold=0.5):
    metricas = []
    matrizes = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline.fit(X_train, y_train)
        proba = pipeline.predict_proba(X_test)[:, 1]
        preds = (proba >= threshold).astype(int)

        metricas.append(calcular_metricas(y_test, preds, proba))
        matrizes.append(confusion_matrix(y_test, preds, labels=[0, 1]))

    media_metricas = pd.DataFrame(metricas).mean()
    media_metricas["threshold"] = threshold

    # matriz de confusão média (tn, fp, fn, tp) somada entre os folds -> total do dataset
    cm_total = np.sum(matrizes, axis=0)
    return media_metricas, cm_total

<h2 id="mlflow">MLflow</h2>

Configuramos o tracking do MLflow no backend de **arquivos** (`mlruns/` na raiz
do projeto) — sem banco de dados. Isso funciona no MLflow 2.x, que é a versão
fixada no `requirements.txt`. Cada estratégia testada vira um *run* dentro do
experimento `churn-etapa-2`, registrando parâmetros e métricas. Assim, os
experimentos de ensemble ficam rastreados no mesmo lugar que os do MLP
(requisito da Etapa 2).

Para inspecionar depois, rode no terminal, na raiz do projeto:

```bash
mlflow ui
```

e acesse `http://127.0.0.1:5000`.

In [ ]:
mlflow.set_tracking_uri((PROJECT_ROOT / "mlruns").as_uri())
mlflow.set_experiment("churn-etapa-2")

def registrar_mlflow(nome, modelo_tipo, metricas, params_extra=None):
    """Registra um run no MLflow com params e métricas de uma estratégia."""
    with mlflow.start_run(run_name=nome):
        mlflow.set_tag("modelo", modelo_tipo)
        mlflow.log_param("estrategia", nome)
        mlflow.log_param("cv_folds", cv.get_n_splits())
        mlflow.log_param("random_state", RANDOM_STATE)
        if params_extra:
            mlflow.log_params(params_extra)
        for chave in ["accuracy", "precision", "recall", "f1", "roc_auc", "threshold"]:
            if chave in metricas:
                mlflow.log_metric(chave, float(metricas[chave]))

Uma pequena função auxiliar para rodar uma estratégia, registrar no MLflow e
devolver as métricas num dicionário pronto para a tabela comparativa:

In [ ]:
def avaliar(nome, pipeline, modelo_tipo, threshold=0.5, params_extra=None):
    media, cm = cv_resultados(pipeline, threshold=threshold)
    registrar_mlflow(nome, modelo_tipo, media, params_extra)
    linha = media.to_dict()
    linha["estrategia"] = nome
    matrizes_confusao[nome] = cm
    print(f"{nome:>22} | f1={linha['f1']:.4f} recall={linha['recall']:.4f} roc_auc={linha['roc_auc']:.4f}")
    return linha

# guarda as matrizes de confusão de cada estratégia para a análise de custo
matrizes_confusao = {}
resultados = []

<h2 id="rf">Random Forest</h2>

Primeiro ensemble: floresta aleatória. Testamos a versão padrão, o ajuste de
threshold, o uso de `class_weight="balanced"` (alternativa nativa das árvores
para desbalanceamento) e SMOTE.

<h3 id="rf-baseline">Baseline</h3>

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", rf),
])

resultados.append(
    avaliar("rf_baseline", rf_pipeline, "RandomForest",
            params_extra={"n_estimators": 300})
)

<h3 id="rf-threshold">Threshold ajustado</h3>

Reduzir o limiar aumenta o recall — o que interessa em churn — ao custo de
precisão. Testamos a mesma faixa usada no MLP.

In [ ]:
for thr in [0.30, 0.35, 0.40, 0.45]:
    resultados.append(
        avaliar(f"rf_threshold_{thr:.2f}", rf_pipeline, "RandomForest",
                threshold=thr, params_extra={"n_estimators": 300})
    )

<h3 id="rf-classweight">Class weight</h3>

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_balanced_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", rf_balanced),
])

resultados.append(
    avaliar("rf_class_weight", rf_balanced_pipeline, "RandomForest",
            params_extra={"n_estimators": 300, "class_weight": "balanced"})
)

<h3 id="rf-smote">SMOTE</h3>

In [ ]:
rf_smote_pipeline = ImbPipeline([
    ("preprocessador", preprocessador),
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("classificador", rf),
])

resultados.append(
    avaliar("rf_smote", rf_smote_pipeline, "RandomForest",
            params_extra={"n_estimators": 300, "sampler": "SMOTE"})
)

<h2 id="gb">Gradient Boosting</h2>

Segundo ensemble: boosting sequencial de árvores. O `GradientBoostingClassifier`
não tem `class_weight`, então tratamos o desbalanceamento por threshold e SMOTE.

<h3 id="gb-baseline">Baseline</h3>

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_STATE,
)

gb_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", gb),
])

resultados.append(
    avaliar("gb_baseline", gb_pipeline, "GradientBoosting",
            params_extra={"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3})
)

<h3 id="gb-threshold">Threshold ajustado</h3>

In [ ]:
for thr in [0.30, 0.35, 0.40, 0.45]:
    resultados.append(
        avaliar(f"gb_threshold_{thr:.2f}", gb_pipeline, "GradientBoosting",
                threshold=thr,
                params_extra={"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3})
    )

<h3 id="gb-smote">SMOTE</h3>

In [ ]:
gb_smote_pipeline = ImbPipeline([
    ("preprocessador", preprocessador),
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("classificador", gb),
])

resultados.append(
    avaliar("gb_smote", gb_smote_pipeline, "GradientBoosting",
            params_extra={"n_estimators": 300, "sampler": "SMOTE"})
)

<h2 id="comparacao">Comparação Consolidada</h2>

Montamos a tabela com todos os ensembles e, quando disponível, carregamos os
resultados do MLP (salvos em `reports/mlp_resultados.csv`) para consolidar tudo
num único ranking. Ordenamos por F1-score, que equilibra precisão e recall.

In [ ]:
ensemble_df = pd.DataFrame(resultados)
ensemble_df["modelo"] = ensemble_df["estrategia"].str.split("_").str[0].map(
    {"rf": "RandomForest", "gb": "GradientBoosting"}
)

# carrega os resultados do MLP, se existirem, para o ranking consolidado
mlp_csv = PROJECT_ROOT / "reports" / "mlp_resultados.csv"
if mlp_csv.exists():
    mlp_df = pd.read_csv(mlp_csv)
    mlp_df["estrategia"] = "mlp_" + mlp_df["estrategia"].astype(str)
    mlp_df["modelo"] = "MLP"
    consolidado = pd.concat([mlp_df, ensemble_df], ignore_index=True)
else:
    print("Aviso: reports/mlp_resultados.csv não encontrado. Mostrando só os ensembles.")
    consolidado = ensemble_df.copy()

colunas = ["modelo", "estrategia", "accuracy", "precision", "recall", "f1", "roc_auc"]
consolidado = consolidado[[c for c in colunas if c in consolidado.columns]]
consolidado = consolidado.sort_values("f1", ascending=False).reset_index(drop=True)
consolidado

<h3 id="custo">Análise de Custo (FP × FN)</h3>

Em churn, os dois tipos de erro têm custos diferentes:

- **Falso negativo (FN)**: o modelo diz que o cliente fica, mas ele cancela.
  Perde-se o cliente sem chance de retê-lo — custo alto (aquisição de um novo
  cliente costuma ser bem mais cara que retenção).
- **Falso positivo (FP)**: o modelo diz que o cliente vai sair, mas ele ficaria.
  Gasta-se uma ação de retenção (desconto, contato) desnecessária — custo menor.

Assumimos, como premissa de negócio a validar com stakeholders, que **um FN custa
5× mais que um FP**. Com as matrizes de confusão acumuladas na validação cruzada
(portanto, sobre todo o dataset), calculamos o custo total de cada estratégia de
ensemble e escolhemos a de menor custo.

In [ ]:
CUSTO_FN = 5   # perder um cliente que ia cancelar
CUSTO_FP = 1   # ação de retenção desnecessária

linhas_custo = []
for nome, cm in matrizes_confusao.items():
    tn, fp, fn, tp = cm.ravel()
    custo = fn * CUSTO_FN + fp * CUSTO_FP
    linhas_custo.append({
        "estrategia": nome,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "custo_total": int(custo),
    })

custo_df = pd.DataFrame(linhas_custo).sort_values("custo_total").reset_index(drop=True)
print(f"Premissa: custo_FN = {CUSTO_FN}, custo_FP = {CUSTO_FP}")
custo_df

> Nota: registramos o custo total de cada estratégia também como métrica no
> MLflow, para que a decisão de negócio fique rastreada junto dos experimentos.

In [ ]:
for _, r in custo_df.iterrows():
    with mlflow.start_run(run_name=f"custo_{r['estrategia']}"):
        mlflow.set_tag("tipo", "analise_custo")
        mlflow.log_params({"custo_fn": CUSTO_FN, "custo_fp": CUSTO_FP})
        mlflow.log_metrics({
            "fp": float(r["fp"]), "fn": float(r["fn"]),
            "custo_total": float(r["custo_total"]),
        })
print("Custos registrados no MLflow.")

<h2 id="conclusao">Conclusão</h2>

Cruzamos duas leituras: o ranking por F1-score (equilíbrio precisão/recall) e o
custo de negócio (FP × FN). A estratégia campeã do ensemble deve ter bom F1 e
recall alto — priorizando capturar quem vai cancelar — sem custo total muito
acima das demais.

In [ ]:
melhor_ensemble = ensemble_df.sort_values("f1", ascending=False).iloc[0]
melhor_por_custo = custo_df.iloc[0]

print("Melhor ensemble por F1-score:")
print(melhor_ensemble[["estrategia", "recall", "f1", "roc_auc"]].to_string())
print()
print("Melhor ensemble por custo de negócio:")
print(melhor_por_custo.to_string())

<h3 id="recomendacao">Recomendação</h3>

> **Preencher após rodar.** Com os números em mãos, indique qual estratégia de
> ensemble foi escolhida e por quê, comparando com o MLP campeão da Etapa 2
> (threshold 0,40 → recall 0,651, F1 0,619, ROC-AUC 0,843). Argumente com base no
> F1/recall **e** na análise de custo. Ajuste a variável `ESTRATEGIA_CAMPEA`
> abaixo para o nome exato da estratégia vencedora antes de exportar.

<h3 id="exportacao">Exportação</h3>

#### Modelo

Retreinamos a pipeline campeã em todo o conjunto de dados e exportamos o artefato
no mesmo formato do notebook do MLP (dicionário com pipeline, threshold e nome do
modelo), para manter a compatibilidade com a etapa de API.

In [ ]:
# ajuste para a estratégia vencedora escolhida na recomendação acima
ESTRATEGIA_CAMPEA = melhor_ensemble["estrategia"]

# mapeia o nome da estratégia -> (pipeline, threshold) para retreino
def resolver_pipeline(nome):
    thr = 0.5
    if "threshold" in nome:
        thr = float(nome.split("_")[-1])
    if nome.startswith("rf_class_weight"):
        return rf_balanced_pipeline, thr
    if nome.startswith("rf_smote"):
        return rf_smote_pipeline, thr
    if nome.startswith("rf"):
        return rf_pipeline, thr
    if nome.startswith("gb_smote"):
        return gb_smote_pipeline, thr
    if nome.startswith("gb"):
        return gb_pipeline, thr
    raise ValueError(f"Estratégia desconhecida: {nome}")

pipeline_campea, threshold_campeao = resolver_pipeline(ESTRATEGIA_CAMPEA)
pipeline_campea.fit(X, y)

artefato = {
    "pipeline": pipeline_campea,
    "threshold": threshold_campeao,
    "model_name": ESTRATEGIA_CAMPEA,
}

(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)
caminho_modelo = PROJECT_ROOT / "models" / f"ensemble_{ESTRATEGIA_CAMPEA}.joblib"
joblib.dump(artefato, caminho_modelo)
print(f"Modelo exportado: {caminho_modelo.name} (threshold={threshold_campeao})")

#### Resultados

Salvamos a tabela de ensembles, a tabela consolidada e a análise de custo em
`reports/`, e as métricas do campeão em `reports/metrics/`, seguindo o mesmo
padrão do notebook do MLP.

In [ ]:
(PROJECT_ROOT / "reports" / "metrics").mkdir(parents=True, exist_ok=True)

ensemble_df.to_csv(PROJECT_ROOT / "reports" / "ensemble_resultados.csv", index=False)
consolidado.to_csv(PROJECT_ROOT / "reports" / "comparativo_consolidado.csv", index=False)
custo_df.to_csv(PROJECT_ROOT / "reports" / "ensemble_custo.csv", index=False)

metricas_campea = ensemble_df[ensemble_df["estrategia"] == ESTRATEGIA_CAMPEA][
    ["accuracy", "precision", "recall", "f1", "roc_auc"]
]
metricas_campea.to_json(
    PROJECT_ROOT / "reports" / "metrics" / f"metricas_ensemble_{ESTRATEGIA_CAMPEA}.json",
    orient="records",
    indent=2,
)
print("Resultados salvos!")